In [1]:
from pathlib import Path

import hdbscan
import numpy as np
import pandas as pd
import umap
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import normalize

from visualization.plot import plot_clustering_on_umap
from visualization.samples_plot import show_clustering_samples
from clustering.consensus import consensus_clustering_from_labels
from clustering.consensus_dbcv import dbcv_per_cluster_consensus

PROJECT_ROOT = Path.cwd().resolve().parent
EMBEDDING_DIR = PROJECT_ROOT / "data" / "glomeruli" / "embeddings"

EMBEDDING_NAME = "densenet_crops_embeddings"

embedding: np.ndarray = np.load(EMBEDDING_DIR / f"{EMBEDDING_NAME}.npy")
csv = pd.read_csv(EMBEDDING_DIR / f"{EMBEDDING_NAME}.csv")
print(embedding.shape[1])
n_samples = embedding.shape[0]

2026-07-06 17:25:35.634876: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


1664


In [2]:
X_reduced_variance = VarianceThreshold(threshold=1e-8).fit_transform(embedding)

embedding_scaled = StandardScaler().fit_transform(X_reduced_variance)

pca_95 = PCA(n_components=0.95, svd_solver="full")
pca_95_embedding = pca_95.fit_transform(embedding_scaled)

print("Shape dopo PCA:", pca_95_embedding.shape)

pca_95_embedding_l2 = normalize(pca_95_embedding, norm="l2")

Shape dopo PCA: (677, 315)


In [ ]:
labels_for_consensus = []
umap_embeddings_for_consensus = []

for seed in range(1, 51):
    umap_embedding = umap.UMAP(
        n_neighbors=int(0.075 * n_samples),  #umap_params["n_neighbors"],
        min_dist=0.05,
        random_state=seed,  #umap_params["min_dist"],
        n_components=20,  #umap_params["n_components"],
        metric="euclidean"  #umap_params["metric"],
    ).fit_transform(pca_95_embedding_l2)

    CLUSTER_SELECTION_METHOD = "eom"

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=round(0.03 * n_samples),  #hdbscan_params["min_cluster_size"],
        min_samples=5,  #round(0.01 * n_samples),  #          hdbscan_params["min_samples"],
        metric="euclidean",  #hdbscan_params["metric"],
        cluster_selection_method=CLUSTER_SELECTION_METHOD  #hdbscan_params["cluster_selection_method"]
    )

    labels = clusterer.fit_predict(umap_embedding)

    labels_for_consensus.append(labels)
    umap_embeddings_for_consensus.append(umap_embedding)


In [3]:
from clustering.score import consensus_param_score, consensus_param_metrics
import pandas as pd
from tqdm.auto import tqdm

from clustering.fit import fit_umap_hdbscan
from clustering.params import make_umap_hdbscan_grid_params

params = make_umap_hdbscan_grid_params(
    n_components_values=[15, 20, 30],
    n_neighbors_values=[round(0.05 * n_samples), round(0.075 * n_samples)],
    min_cluster_size_values=[round(0.02 * n_samples), round(0.03 * n_samples)],
    min_samples_values=[3, 5, round(0.01 * n_samples)]
)

candidate_results = []

for config_id, param in enumerate(tqdm(params, desc="Grid search consensus")):
    labels_for_consensus = []
    umap_embeddings_for_consensus = []

    for seed in range(1, 21):
        labels, umap_embedding = fit_umap_hdbscan(
            X=pca_95_embedding_l2,
            seed=None,
            n_components=param["n_components"],
            n_neighbors=param["n_neighbors"],
            min_cluster_size=param["min_cluster_size"],
            min_samples=param["min_samples"],
        )

        labels_for_consensus.append(labels)
        umap_embeddings_for_consensus.append(umap_embedding)

    consensus_labels, probabilities = consensus_clustering_from_labels(
        labels_for_consensus,
        max_noise_frequency=0.33,
        min_consensus_strength=0.70,
    )

    dbcv_consensus = dbcv_per_cluster_consensus(
        consensus_labels,
        umap_embeddings_for_consensus,
    )

    score = consensus_param_score(
        dbcv_df=dbcv_consensus,
        consensus_labels=consensus_labels,
        max_noise=0.35,
    )

    metrics = consensus_param_metrics(
        dbcv_df=dbcv_consensus,
        consensus_labels=consensus_labels,
    )

    candidate_results.append({
        "config_id": config_id,
        **param,
        "score": score,
        **metrics,
        "consensus_labels": consensus_labels,
        "probabilities": probabilities,
        "dbcv_df": dbcv_consensus,
    })

Grid search consensus:   0%|          | 0/36 [00:00<?, ?it/s]

In [4]:
results_df = pd.DataFrame([
    {
        k: v
        for k, v in result.items()
        if k not in ["consensus_labels", "probabilities", "dbcv_df"]
    }
    for result in candidate_results
])

results_df = results_df.sort_values("score", ascending=False).reset_index(drop=True)

In [5]:
best_config_id = int(results_df.iloc[0]["config_id"])
best_result = candidate_results[best_config_id]

best_consensus_labels = best_result["consensus_labels"]
best_probabilities = best_result["probabilities"]
best_dbcv_df = best_result["dbcv_df"]

best_result_summary = {
    k: v
    for k, v in best_result.items()
    if k not in ["consensus_labels", "probabilities", "dbcv_df"]
}

best_result_summary

{'config_id': 8,
 'min_cluster_size': 14,
 'min_samples': 5,
 'n_components': 20,
 'n_neighbors': 34,
 'score': 0.7369232862276749,
 'n_clusters': 4,
 'noise_ratio': 0.20974889217134415,
 'global_dbcv_mean': 0.7430473978312291,
 'global_dbcv_std': 0.02449644641421668,
 'min_cluster_dbcv_median': 0.5074920295693095,
 'mean_cluster_dbcv_median': 0.7124587275926273}

In [7]:
best_cluster_dbcv = best_result["dbcv_df"]

display(
    best_cluster_dbcv[
        [
            "cluster",
            "size",
            "dbcv_mean",
            "dbcv_std",
            "dbcv_min",
            "dbcv_max",
            "dbcv_median",
            "n_runs_valid",
        ]
    ].sort_values("cluster").reset_index(drop=True)
)

,cluster,size,dbcv_mean,dbcv_std,dbcv_min,dbcv_max,dbcv_median,n_runs_valid
0,-1,142,NaN,NaN,NaN,NaN,NaN,0
1,0,129,0.483279,0.089043,0.214320,0.594898,0.507492,20
2,1,333,0.839156,0.012084,0.802990,0.856886,0.841755,20
3,2,29,0.647919,0.080835,0.465258,0.748588,0.661410,20
4,3,44,0.839973,0.019339,0.807934,0.878856,0.839178,20


In [ ]:


consensus_labels, probabilities = consensus_clustering_from_labels(
    labels_for_consensus,
    max_noise_frequency=0.33,
    min_consensus_strength=0.70
)

In [ ]:


dbcv_consensus = dbcv_per_cluster_consensus(
    consensus_labels,
    umap_embeddings_for_consensus
)

dbcv_consensus

In [ ]:
plot_clustering_on_umap(
    pca_95_embedding_l2,
    consensus_labels,
    metric="euclidean",
    min_dist=0.05,
    n_neighbors=int(0.075 * n_samples)  #umap_params["n_neighbors"],
)

In [ ]:
image_paths = csv["image_path"].tolist()

sample_figure, sample_axes = show_clustering_samples(
    labels=consensus_labels,
    probabilities=probabilities,
    image_paths=image_paths,
    x=10,
    base_dir=PROJECT_ROOT,
    image_size=2.0,
)

In [ ]:
from sklearn.metrics import pairwise_distances


def pair_rank_report(X, idx_a, idx_b, metric="cosine", name="space"):
    """
    Dice quanto idx_b ﷿﷿﷿﷿﷿﷿ vicino a idx_a nello spazio X.
    rank = 1 significa vicino pi﷿﷿﷿﷿﷿﷿ vicino.
    rank alto significa lontano.
    """
    D = pairwise_distances(X, X[[idx_a]], metric=metric).ravel()

    order = np.argsort(D)
    order = order[order != idx_a]  # rimuove s﷿﷿﷿﷿﷿﷿ stesso

    rank_b = np.where(order == idx_b)[0][0] + 1
    dist_ab = D[idx_b]

    percent_rank = rank_b / (len(X) - 1) * 100

    print(f"\n{name}")
    print(f"metric: {metric}")
    print(f"distanza {idx_a} -> {idx_b}: {dist_ab:.4f}")
    print(f"rank del secondo rispetto al primo: {rank_b}/{len(X) - 1}")
    print(f"percentile rank: {percent_rank:.2f}%")

    return {
        "space": name,
        "distance": dist_ab,
        "rank": rank_b,
        "percent_rank": percent_rank,
    }

In [ ]:
ID_SCLEROTIC = 4
ID_OPEN = 3

In [ ]:
# Distanza originale tra due punti nello spazio embedding
report_pre = pair_rank_report(
    embedding,
    ID_SCLEROTIC,
    ID_OPEN,
    metric="cosine",
    name="PRE-UMAP"
)

post_variance_threshold = pair_rank_report(
    X_reduced_variance,
    ID_SCLEROTIC,
    ID_OPEN,
    metric="cosine",
    name="POST-VarianceThreshold"
)

post_pca = pair_rank_report(
    pca_95_embedding_l2,
    ID_SCLEROTIC,
    ID_OPEN,
    metric="cosine",
    name="POST-PCA"
)

post_umap = pair_rank_report(
    umap_embedding,
    ID_SCLEROTIC,
    ID_OPEN,
    metric="euclidean",
    name="POST-UMAP"
)